# 4. Conclusiones, Limitaciones y Reproducibilidad

Ventanas de entrada seleccionadas por RMSE TEST: **BTCUSDT = 28**, **ETHUSDT = 28**, **BNBUSDT = 7**, **XRPUSDT = 28**, **SOLUSDT = 7**. El horizonte permanece en 7 horas y la volatilidad se calcula con 30 retornos horarios. La conclusión final que sigue procede de la celda 26 del notebook maestro.

### Resultados principales

La comparación de los resultados finales de TEST identifica una ventana de entrada de **28 observaciones horarias para BTCUSDT, ETHUSDT y XRPUSDT**, y de **7 observaciones horarias para BNBUSDT y SOLUSDT**. No existe una única ventana de entrada óptima para todos los activos dentro de las configuraciones evaluadas: ampliar la historia de entrada no garantiza un menor error. Cada criptomoneda se selecciono por separado, sin elegir un ganador global entre series distintas.

En todos los activos, para su ventana seleccionada, el **RMSE medio de TEST aumenta progresivamente desde h=1 hasta h=7**. Esto indica un deterioro de la precisión conforme aumenta el horizonte de pronóstico en la muestra evaluada; no implica que cada observacion individual o cada fold presente necesariamente el mismo patron.

La tabla reproduce los valores almacenados en `results/best_window_by_symbol.csv`, sin redondeo adicional. Las medias resumen los cinco folds y, para las metricas de error, los siete horizontes. `RMSE_std` es la desviación estándar muestral entre folds. MAE y RMSE se expresan en unidades originales de volatility, MSE en unidades al cuadrado y MAPE en porcentaje. `RMSE_mean` promedia los RMSE por horizonte; no es la raiz del MSE promedio.

| symbol | best_input_window | RMSE_mean | RMSE_std | MAE_mean | MAPE_mean | MSE_mean | BDS_pvalue_mean |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| BTCUSDT | 28 | 0.0007312671061778 | 0.0001845722374319 | 0.0004945991773445 | 14.31958954138831 | 6.042626750259671e-07 | 0.1808025235834009 |
| ETHUSDT | 28 | 0.0009595219011006 | 0.0001494570390571 | 0.0005771923300969 | 10.828846192907438 | 1.0008363265460483e-06 | 0.3073685539671471 |
| BNBUSDT | 7 | 0.0008724995689284 | 0.0002632498376451 | 0.0005546200873572 | 12.592242972856562 | 8.826787758492946e-07 | 0.2024245322497591 |
| XRPUSDT | 28 | 0.0013666986629243 | 0.0002812370954898 | 0.0007905575383531 | 13.173790838842104 | 2.0632155755113024e-06 | 0.1365720924102251 |
| SOLUSDT | 7 | 0.0012514238195019 | 0.0002710002728802 | 0.0007845884011077 | 10.48172877386926 | 1.7679771706002514e-06 | 0.2996031567673319 |

Como métrica complementaria de regresión temporal, las configuraciones seleccionadas alcanzaron R² medios TEST de 0.897940 para BTCUSDT, 0.892582 para ETHUSDT, 0.902511 para BNBUSDT, 0.896526 para XRPUSDT y 0.906606 para SOLUSDT. El R² disminuyó progresivamente al aumentar el horizonte de h=1 a h=7. Estos valores elevados deben interpretarse considerando la persistencia y el solapamiento inherentes a la volatilidad rolling de 30 horas y no constituyen por sí solos evidencia de ausencia de leakage ni garantía de desempeño operativo futuro.

### Interpretación del BDS

`BDS_pvalue_mean` resume descriptivamente los p-values obtenidos sobre los residuos de TEST del horizonte h=1. **El promedio de p-values no constituye una prueba combinada ni permite decidir por si solo sobre la hipótesis nula.** La inferencia debe realizarse por fold, consultando `results/bds_best_window_by_fold.csv`.

La hipótesis nula del test BDS corresponde a una secuencia independiente e identicamente distribuida. Para cada fold, p > 0,05 significa que **no se rechaza H0 del test BDS**; p <= 0,05 significa que **se rechaza H0 al nivel del 5 %**. No rechazar H0 no demuestra independencia ni establece que el modelo sea bueno. Estos diagnosticos deben complementarse con los errores de pronóstico y su estabilidad temporal.

### Limitaciones

- El MLP recibe únicamente historia de `volatility`. No incorpora volumen, precio como entrada directa, noticias ni otras variables exógenas. El precio de cierre se utilizo para construir los retornos y el target, pero no se proporciona como feature independiente al MLP.
- El target es volatilidad historica calculada con una desviación estándar móvil de **30 observaciones horarias** de retornos logarítmicos, con `ddof=1`, sin anualizar. Es una medida suavizada; el solapamiento entre ventanas contribuye a su persistencia.
- El horizonte se limita a **7 horas**. Estos resultados no validan pronósticos a plazos superiores.
- El desempeño puede variar en otros periodos y regimenes de mercado. Los folds muestran variacion temporal y comparten historia de entrenamiento, por lo que no son replicas independientes.
- La partición nativa de `tsxv.splitTrainValTest.split_train_val_test_groupKFold` fue evaluada y tuvo que adaptarse: intercalaba train, validation y test a lo largo de la serie y permitia ventanas sobre discontinuidades. Se construyeron ventanas dentro de cada `segment_id` y cinco folds cronológicos expansivos con purga en sus fronteras, para evitar leakage temporal. Esta adaptación es explicita y no se atribuye a la partición nativa.
- La ventana se eligio retrospectivamente por RMSE TEST. Por tanto, ese TEST ya no constituye una evaluacion independiente posterior a la seleccion; haria falta un periodo externo no utilizado para escoger la configuracion.
- No se demuestra superioridad frente a persistencia u otros baselines, que no se compararon en esta etapa. Tampoco se estudio la sensibilidad a multiples semillas o arquitecturas. Un menor RMSE absoluto entre activos no determina por si solo cual serie se pronostica mejor, ya que sus escalas difieren.

### Reproducibilidad

- **Datos:** Binance Spot, velas de frecuencia **1h**, reutilizando `data/processed/crypto_binance_master_1h.csv`, sin descargar nuevamente. Los tiempos `open_time` y `close_time` se interpretan en UTC mediante `pd.to_datetime(..., format="mixed", utc=True)` al cargar el CSV.
- **Activos:** BTCUSDT, ETHUSDT, BNBUSDT, XRPUSDT y SOLUSDT, procesados por separado y ordenados temporalmente.
- **Target:** `volatility`, desviación estándar móvil de 30 retornos horarios; retornos y rolling calculados dentro de cada simbolo y segmento continuo. Se excluyen solo las filas no utilizables por ese calculo.
- **Entradas y salidas:** `INPUT_WINDOWS = [7, 14, 21, 28]`, `N_STEPS_JUMP = 1` y `N_STEPS_FORECAST = 7`. Cada muestra usa solo historia de volatility y predice simultaneamente los siete valores futuros. La ventana completa X+y permanece en un mismo segmento.
- **Validación:** cinco folds cronológicos expansivos por configuracion, obtenidos a partir de siete bloques consecutivos de ventanas validas y la purga documentada. Se cumple `train_end < validation_start` y `validation_end < test_start`, considerando las fechas completas de X+y, sin shuffle de las particiónes. Las fechas y tamanos se conservan en `results/timeseries_cv_structure.csv`.
- **Escalamiento:** dos `StandardScaler` independientes por fold, para X e y, ajustados exclusivamente con train. Validation y test solo se transforman. Las predicciones vuelven a la escala original mediante `scaler_y.inverse_transform` antes de calcular metricas.
- **Modelo:** `MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, random_state=RANDOM_STATE + fold, shuffle=False, early_stopping=False)`, con `RANDOM_STATE = 42` y folds numerados de 1 a 5. Cada modelo tiene siete salidas; no se entrena un modelo separado por horizonte. Se conservan los demas parametros predeterminados de la version utilizada.
- **Entorno registrado:** Python 3.10.21 en `vol_venv`, scikit-learn 1.7.2 y statsmodels 0.15.0. La semilla fija permite reproducir el procedimiento; conservar versiones, datos y configuracion numerica ayuda a reproducir los resultados.
- **Evaluación:** MAE, MSE, RMSE y MAPE por horizonte y conjunto; MAPE utiliza `epsilon=1e-8` y se expresa en porcentaje. BDS se aplica a residuos TEST h=1 con `max_dim=2` y `distance=1.5`. Se conservan los tres CSV originales de metricas, los diagnosticos, las predicciones NPZ y las tablas y figuras finales.

Esta seccion se elaboro exclusivamente a partir de los resultados finales existentes. No se reentrenaron modelos, no se modificaron los folds y no se alteraron CSV ni figuras para redactar estas conclusiones.


### Limitaciones adicionales

El alcance se limita a cinco activos de un exchange, Binance Spot, durante un período específico. No se presupone representatividad de otros exchanges, criptoactivos o regímenes. Las asociaciones y los pronósticos no establecen causalidad y los resultados no constituyen recomendación financiera.

Se mantienen las limitaciones ya documentadas: ausencia de variables exógenas, horizonte limitado a siete horas, target definido como volatilidad rolling 30h y adaptación explícita de la validación temporal. El EDA complementario describe perfiles temporales y el segmento continuo mayor de cada activo; no demuestra estacionariedad global ni modifica las ventanas seleccionadas: BTCUSDT = 28, ETHUSDT = 28, BNBUSDT = 7, XRPUSDT = 28 y SOLUSDT = 7.


## Matriz de cumplimiento de la guía

| Criterio | Estado | Evidencia | Justificación |
| --- | --- | --- | --- |
| Base de datos | CUMPLIDO | Base de datos, apartados originales y 1.6–1.11 | Fuente, preparación y alcance documentados. |
| Definición del problema | CUMPLIDO | Base de datos 1.6 | Volatilidad futura t+1 a t+7. |
| Justificación del dataset | CUMPLIDO | Base de datos 1.7 | Datos reales horarios, API pública y reproducibilidad. |
| Fuente | CUMPLIDO | Fuente, frecuencia y período | Binance Spot y caché CSV. |
| Diccionario de variables | CUMPLIDO | Variables originales y símbolo; target | Campos originales y variables derivadas explicados. |
| Tamaño muestral | CUMPLIDO | Base de datos 1.8 | 289.094 filas originales; 266.060 útiles. |
| Calidad | CUMPLIDO | Base de datos 1.9; EDA 2.2–2.3 | NaN, duplicados, huecos, segmentos e infinitos. |
| Ética | CUMPLIDO | Base de datos 1.11 | Datos de mercado sin información personal. |
| 2.1/2.3 Variable objetivo actual | CUMPLIDO | EDA 2.3 | Volatilidad rolling de 30 retornos horarios. |
| Análisis unidimensional | CUMPLIDO | EDA 2.4 | Descriptivos, histogramas y boxplots existentes. |
| Análisis bidimensional | CUMPLIDO | EDA 2.4 y 2.4.2 | Correlaciones contemporáneas y su interpretación. |
| Análisis multivariado clásico | NO APLICA | EDA 2.9; modelado 3.1 | El predictor modelado es una serie univariada de volatility con rezagos temporales; siete salidas no implican variables exógenas múltiples. |
| 2.5 Leakage | CUMPLIDO | EDA 2.5 | Disponibilidad en t, purga y separación temporal. |
| 2.6 Temporal | CUMPLIDO | EDA 2.6.1–2.6.7 | ACF existente, perfiles, ADF/KPSS y PACF nuevos. |
| 2.7 Espacial | NO APLICA | EDA 2.7 | Sin coordenadas ni unidades espaciales. |
| 2.8 Espacio-temporal | NO APLICA | EDA 2.8 | Existe tiempo, pero no dimensión espacial. |
| 2.9 Preprocesamiento | CUMPLIDO | EDA 2.9 | Transformaciones y separación existentes conservadas. |
| MLP multisalida | CUMPLIDO | Modelado 3.2 | Arquitectura (64,32); 100 modelos ya entrenados. |
| Ventanas 7/14/21/28 | CUMPLIDO | Modelado 3.1 | Historia de volatility por segmento. |
| Horizonte 7 | CUMPLIDO | Modelado 3.1–3.2 | Siete salidas simultáneas. |
| Folds cronológicos | CUMPLIDO | Modelado 3.1 | Cinco folds expansivos con purga. |
| Scaling train-only | CUMPLIDO | Modelado 3.2 | Dos StandardScaler ajustados exclusivamente con train. |
| MAE | CUMPLIDO | Modelado 3.2–3.3 | Resultados guardados por horizonte y fold. |
| MSE | CUMPLIDO | Modelado 3.2–3.3 | Resultados guardados en unidades al cuadrado. |
| RMSE | CUMPLIDO | Modelado 3.2–3.3 | Media de RMSE por horizonte y selección retrospectiva explícita. |
| MAPE | CUMPLIDO | Modelado 3.2–3.3 | Porcentaje, epsilon=1e-8 y limitación cerca de cero. |
| BDS | CUMPLIDO | Modelado 3.2 y aclaración final | Residuos TEST h=1; inferencia por fold. |
| Reproducibilidad | CUMPLIDO | Conclusiones, Reproducibilidad; README.md | Entorno original, caché, artefactos, semilla y comandos documentados. |
| R² | CUMPLIDO | Modelado, Coeficiente de determinación R² | Métrica complementaria de regresión temporal calculada sobre predicciones TEST almacenadas. |
